# Exercise 3 — generate_github_readme

GitHub's profile README is the most-visited page in your personal brand. When someone visits `github.com/yourusername`, they see the README from the repository named `yourusername/yourusername`. `generate_github_readme` builds this document from your PortfolioConfig: a greeting, bio, project list, deduplicated tech stack, and contact info.

In [ ]:
import pathlib, tempfile, os
from collections import Counter
from dataclasses import dataclass, field

@dataclass
class ProjectEntry:
    name: str; tagline: str; description: str; tech_stack: list
    github_url: str = ""; demo_url: str = ""; category: str = "AI Engineering"
    highlights: list = field(default_factory=list)

@dataclass
class PortfolioConfig:
    owner_name: str; title: str; bio: str; email: str; github_username: str
    linkedin_url: str = ""; projects: list = field(default_factory=list)

_P1 = ProjectEntry(
    name        = "AI Trading Bot",
    tagline     = "Paper-trading bot with sentiment + technical signals.",
    description = "Built over Days 89-96, this bot fetches OHLCV data, computes "
                  "technical indicators, scores news headlines with an LLM, applies "
                  "stop-loss and drawdown controls, and logs results daily.",
    tech_stack  = ["Python", "pandas", "Ollama", "SQLite"],
    github_url  = "https://github.com/testuser/ai-trading-bot",
    category    = "Finance",
    highlights  = ["Fully automated daily paper-trading loop",
                   "Kelly Criterion position sizing", "Stop-loss + drawdown gating"],
)
_P2 = ProjectEntry(
    name        = "Ops Agent",
    tagline     = "Autonomous multi-step ops agent with guardrails.",
    description = "Agent loop with tool routing, human-in-the-loop approval gates, "
                  "and task queue persistence.",
    tech_stack  = ["Python", "Ollama", "ChromaDB"],
    category    = "AI Agents",
    highlights  = ["Handles 5 operations autonomously", "Approval gate for destructive ops"],
)
_P3 = ProjectEntry(
    name        = "RAG Chatbot",
    tagline     = "Q&A chatbot grounded in your documents.",
    description = "Retrieval-augmented generation over a personal knowledge base.",
    tech_stack  = ["Python", "ChromaDB", "Ollama", "FastAPI"],
    github_url  = "https://github.com/testuser/rag-chatbot",
    demo_url    = "https://rag-chatbot.example.com",
    category    = "Text AI",
)

_CFG = PortfolioConfig(
    owner_name      = "Jane Doe",
    title           = "AI Engineer",
    bio             = "I build practical AI applications with Python. "
                      "100 days of AI engineering, shipped.",
    email           = "jane@example.com",
    github_username = "janedoe",
    linkedin_url    = "https://linkedin.com/in/janedoe",
    projects        = [_P1, _P2, _P3],
)

def generate_github_readme(config):
    """Generate a GitHub profile README.md.

    Structure:
      # Hi, I'm {config.owner_name}
      {config.bio}
      ## What I Build
      I'm an **{config.title}** focused on ...
      ## Projects
      - **[Project Name](url)** — tagline
      ## Tech Stack
      tech1 · tech2 · tech3 ...  (unique, order-preserved, max 8)
      ## Contact
      - Email: [email](mailto:email)
      - LinkedIn: url  (only if linkedin_url is set)

    Returns:
        str — Markdown starting with "# Hi, I'm {owner_name}"
    """
    project_lines = "\n".join(
        f"- **[{p.name}]({p.github_url or '#'})** — {p.tagline}"
        for p in config.projects
    )
    all_tech = []
    for p in config.projects: all_tech.extend(p.tech_stack)
    unique_tech = list(dict.fromkeys(all_tech))[:8]   # deduplicate, preserve order
    tech_line = " · ".join(unique_tech)
    linkedin_line = f"- LinkedIn: {config.linkedin_url}\n" if config.linkedin_url else ""
    # TODO: assemble and return the Markdown string
    return ""


### Checks

In [ ]:
checks = 0

# 1 — starts with # Hi, I'm {owner_name}
try:
    readme = generate_github_readme(_CFG)
    expected_start = f"# Hi, I'm {_CFG.owner_name}"
    assert readme.startswith(expected_start),         f"expected '{expected_start}', got: {readme[:50]!r}"
    checks += 1; print(f"✅ 1 README starts with '# Hi, I'm {_CFG.owner_name}'")
except Exception as e:
    print("❌ 1:", e)

# 2 — bio and ## sections present
try:
    readme = generate_github_readme(_CFG)
    assert _CFG.bio in readme, "bio not in README"
    for section in ["## What I Build", "## Projects", "## Tech Stack", "## Contact"]:
        assert section in readme, f"missing section {section}"
    checks += 1; print("✅ 2 bio and all ## sections present")
except Exception as e:
    print("❌ 2:", e)

# 3 — all project names appear
try:
    readme = generate_github_readme(_CFG)
    for p in _CFG.projects:
        assert p.name in readme, f"project {p.name!r} not in README"
    checks += 1; print(f"✅ 3 all {len(_CFG.projects)} project names in README")
except Exception as e:
    print("❌ 3:", e)

# 4 — tech stack: deduplicated, order preserved, max 8
try:
    readme = generate_github_readme(_CFG)
    # Collect unique tech manually to verify
    all_t = []
    for p in _CFG.projects: all_t.extend(p.tech_stack)
    unique = list(dict.fromkeys(all_t))[:8]
    for t in unique:
        assert t in readme, f"tech {t!r} not in README"
    # Python appears in P1 and P2 — check it appears only once in the tech line
    tech_section = readme.split("## Tech Stack")[1].split("##")[0]
    assert tech_section.count("Python") == 1,         f"'Python' should appear once in tech stack, found {tech_section.count('Python')}"
    checks += 1; print("✅ 4 tech stack deduplicated; each item appears once")
except Exception as e:
    print("❌ 4:", e)

# 5 — email present; LinkedIn only if set; absent when not set
try:
    readme = generate_github_readme(_CFG)
    assert _CFG.email        in readme, "email not in README"
    assert _CFG.linkedin_url in readme, "linkedin_url not in README"
    cfg2 = PortfolioConfig("A","E","B","a@b.com","gh")  # no linkedin
    readme2 = generate_github_readme(cfg2)
    assert "LinkedIn" not in readme2, "LinkedIn should not appear without linkedin_url"
    checks += 1; print("✅ 5 email + LinkedIn when set; LinkedIn absent when not set")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
